# Documented English edition

This notebook is the reviewed English edition of `exercicios_computacao_quantica/codigo_shor_erro_critico_pauli_gates.ipynb`. The original file, metadata, and historical outputs are preserved under `codes_obsolete/`. Stored outputs were cleared from this edition; execute the cells sequentially with the declared kernel.


## Dependencies

Import the numerical, visualization, and quantum-computing libraries used below.

In [ ]:
# Purpose: Import the numerical, visualization, and quantum-computing libraries used below.
# ============================================================
# Explanation translated; consult the archived notebook for the original wording.
# Errorrs controlados: X, Y e Z
# ============================================================
# Explanation translated; consult the archived notebook for the original wording.
# !pip install qiskit qiskit-aer pandas matplotlib pylatexenc

from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector, SparsePauliOp, state_fidelity
from qiskit_aer import AerSimulator
import pandas as pd

# ------------------------------------------------------------
# Functions auxiliares
# ------------------------------------------------------------

def pauli_label(pauli, qubits, n = 9):
    """English documentation for this computational helper."""
    label = ["I"] * n
    for q in qubits:
        label[n - 1 - q] = pauli
    return "".join(label)


def stabilizer_operator(label):
    """English documentation for this computational helper."""
    return SparsePauliOp.from_list([(label, 1.0)])


def expectation_sign(state, label, tol=1e-8):
    """
    Calcula o sinal esperado de um estabilizador.
    Retorna +1 ou -1.
    """
    value = state.expectation_value(stabilizer_operator(label)).real
    if value >= -tol:
        return +1
    return -1


# ------------------------------------------------------------
# Explanation translated; consult the archived notebook for the original wording.
# ------------------------------------------------------------

def prepare_initial_state(qc, theta=0.73, phi=0.41):
    """English documentation for this computational helper."""
    qc.ry(theta, 0)
    qc.rz(phi, 0)
    qc.barrier()


def encode_shor(qc):
    """English documentation for this computational helper."""
    # Explanation translated; consult the archived notebook for the original wording.
    qc.cx(0, 3)
    qc.cx(0, 6)
    # Explanation translated; consult the archived notebook for the original wording.
    qc.h(0)
    qc.h(3)
    qc.h(6)
    # Explanation translated; consult the archived notebook for the original wording.
    qc.cx(0, 1)
    qc.cx(0, 2)
    qc.cx(3, 4)
    qc.cx(3, 5)
    qc.cx(6, 7)
    qc.cx(6, 8)
    qc.barrier()


def build_encoded_circuit():
    """English documentation for this computational helper."""
    qc = QuantumCircuit(N_QUBITS)
    prepare_initial_state(qc)
    encode_shor(qc)
    return qc


# ------------------------------------------------------------
# Explanation translated; consult the archived notebook for the original wording.
# ------------------------------------------------------------

def apply_pauli_error(qc, error_type, error_qubit):
    """English documentation for this computational helper."""
    error_type = error_type.upper()
    if error_type == "X":
        qc.x(error_qubit)
    elif error_type == "Y":
        qc.y(error_qubit)
    elif error_type == "Z":
        qc.z(error_qubit)
    else:
        raise ValueErrorr("O erro deve ser 'X', 'Y' ou 'Z'.")
    qc.barrier()


def build_error_circuit(error_type, error_qubit):
    """English documentation for this computational helper."""
    qc = build_encoded_circuit()
    apply_pauli_error(qc, error_type, error_qubit)
    return qc


# ------------------------------------------------------------
# Explanation translated; consult the archived notebook for the original wording.
# ------------------------------------------------------------

def compute_syndrome(state):
    """
    Calcula a syndrome associada aos estabilizadores do Shor code.
    """
    syndrome = {}
    for name, label in STABILIZERS.items():
        syndrome[name] = expectation_sign(state, label)
    return syndrome


def infer_x_error_from_syndrome(syndrome):
    """English documentation for this computational helper."""
    block_syndromes = [
        (syndrome["Z0Z1"], syndrome["Z1Z2"]),
        (syndrome["Z3Z4"], syndrome["Z4Z5"]),
        (syndrome["Z6Z7"], syndrome["Z7Z8"]),
    ]
    for block_index, pair in enumerate(block_syndromes):
        base = 3 * block_index
        if pair == (-1, +1):
            return base
        if pair == (-1, -1):
            return base + 1
        if pair == (+1, -1):
            return base + 2
    return None


def infer_z_error_from_syndrome(syndrome):
    """English documentation for this computational helper."""
    pair = (
        syndrome["X0X1X2X3X4X5"],
        syndrome["X3X4X5X6X7X8"],
    )
    if pair == (-1, +1):
        return 0  # Explanation translated; consult the archived notebook for the original wording.
    if pair == (-1, -1):
        return 3  # Explanation translated; consult the archived notebook for the original wording.
    if pair == (+1, -1):
        return 6  # Explanation translated; consult the archived notebook for the original wording.
    return None


def build_recovery_circuit(syndrome):
    """English documentation for this computational helper."""
    recovery = QuantumCircuit(N_QUBITS)
    x_qubit = infer_x_error_from_syndrome(syndrome)
    z_qubit = infer_z_error_from_syndrome(syndrome)
    if x_qubit is not None:
        recovery.x(x_qubit)
    if z_qubit is not None:
        recovery.z(z_qubit)
    return recovery, x_qubit, z_qubit


# ------------------------------------------------------------
# Explanation translated; consult the archived notebook for the original wording.
# ------------------------------------------------------------

def run_shor_case(error_type, error_qubit):
    """English documentation for this computational helper."""
    # Explanation translated; consult the archived notebook for the original wording.
    ideal_circuit = build_encoded_circuit()
    ideal_state = Statevector.from_instruction(ideal_circuit)
    # Explanation translated; consult the archived notebook for the original wording.
    error_circuit = build_error_circuit(error_type, error_qubit)
    error_state = Statevector.from_instruction(error_circuit)
    # Explanation translated; consult the archived notebook for the original wording.
    syndrome = compute_syndrome(error_state)
    # Explanation translated; consult the archived notebook for the original wording.
    recovery_circuit, x_recovery, z_recovery = build_recovery_circuit(syndrome)
    corrected_state = error_state.evolve(recovery_circuit)
    # Fidelidades
    fidelity_before = state_fidelity(ideal_state, error_state)
    fidelity_after = state_fidelity(ideal_state, corrected_state)
    return {
        "erro_aplicado": error_type,
        "qubit_afetado": error_qubit,
        "recuperacao_X_no_qubit": x_recovery,
        "recuperacao_Z_no_qubit": z_recovery,
        "fidelidade_antes": fidelity_before,
        "fidelidade_depois": fidelity_after,
        "sindrome": syndrome,
        "circuito_com_erro": error_circuit,
        "circuito_recuperacao": recovery_circuit,
    }

## Error syndrome

Define or measure stabilizers used to diagnose an encoded Pauli error.

In [ ]:
# Purpose: Define or measure stabilizers used to diagnose an encoded Pauli error.
N_QUBITS = 9

# ------------------------------------------------------------
# Estabilizadores do Shor code
# ------------------------------------------------------------

STABILIZERS = {
    # Explanation translated; consult the archived notebook for the original wording.
    "Z0Z1": pauli_label("Z", [0, 1]),
    "Z1Z2": pauli_label("Z", [1, 2]),

    "Z3Z4": pauli_label("Z", [3, 4]),
    "Z4Z5": pauli_label("Z", [4, 5]),

    "Z6Z7": pauli_label("Z", [6, 7]),
    "Z7Z8": pauli_label("Z", [7, 8]),

    # Explanation translated; consult the archived notebook for the original wording.
    "X0X1X2X3X4X5": pauli_label("X", [0, 1, 2, 3, 4, 5]),
    "X3X4X5X6X7X8": pauli_label("X", [3, 4, 5, 6, 7, 8]),
}

# ------------------------------------------------------------
# Experimentos principais: X, Y e Z
# ------------------------------------------------------------

cases = [
    ("X", 4),
    ("Y", 4),
    ("Z", 4),
]
results = []
for error_type, error_qubit in cases:
    output = run_shor_case(error_type, error_qubit)
    results.append({
        "Error": output["erro_aplicado"],
        "English diagnostic label": output["qubit_afetado"],
        "English diagnostic label": output["recuperacao_X_no_qubit"],
        "English diagnostic label": output["recuperacao_Z_no_qubit"],
        "Fidelidade antes": output["fidelidade_antes"],
        "Fidelidade depois": output["fidelidade_depois"],
    })
df_results = pd.DataFrame(results)
df_results

## Error syndrome

Define or measure stabilizers used to diagnose an encoded Pauli error.

In [ ]:
# Purpose: Define or measure stabilizers used to diagnose an encoded Pauli error.
# ------------------------------------------------------------
# Estabilizadores do Shor code
# ------------------------------------------------------------

STABILIZERS = {
    # Explanation translated; consult the archived notebook for the original wording.
    "Z0Z1": pauli_label("Z", [0, 1]),
    "Z1Z2": pauli_label("Z", [1, 2]),
    "Z3Z4": pauli_label("Z", [3, 4]),
    "Z4Z5": pauli_label("Z", [4, 5]),
    "Z6Z7": pauli_label("Z", [6, 7]),
    "Z7Z8": pauli_label("Z", [7, 8]),
    # Explanation translated; consult the archived notebook for the original wording.
    "X0X1X2X3X4X5": pauli_label("X", [0, 1, 2, 3, 4, 5]),
    "X3X4X5X6X7X8": pauli_label("X", [3, 4, 5, 6, 7, 8]),
}

# ------------------------------------------------------------
# Experimentos principais: X, Y e Z
# ------------------------------------------------------------

cases = [
    ("X", 4),
    ("Y", 4),
    ("Z", 4),
]
results = []
for error_type, error_qubit in cases:
    output = run_shor_case(error_type, error_qubit)
    results.append({
        "Error": output["erro_aplicado"],
        "English diagnostic label": output["qubit_afetado"],
        "English diagnostic label": output["recuperacao_X_no_qubit"],
        "English diagnostic label": output["recuperacao_Z_no_qubit"],
        "Fidelidade antes": output["fidelidade_antes"],
        "Fidelidade depois": output["fidelidade_depois"],
    })
df_results = pd.DataFrame(results)
df_results

## Simulation

Transpile the circuit for the selected simulator and execute the resulting experiment.

In [ ]:
# Purpose: Prepare and test the nine-qubit Shor error-correcting code.
# Explanation translated; consult the archived notebook for the original wording.
example = run_shor_case("X", 4)
example["circuito_com_erro"].draw("mpl", fold=120)

## Shor-code experiment

Prepare and test the nine-qubit Shor error-correcting code.

In [ ]:
# Purpose: Render the current mathematical object, circuit, or numerical result.
example["circuito_recuperacao"].draw("mpl", fold=120)

## Measurement and counts

Measure the circuit or inspect the finite-shot outcome distribution.

In [ ]:
# Purpose: Measure the circuit or inspect the finite-shot outcome distribution.
simulator = AerSimulator()

full_circuit = example["circuito_com_erro"].compose(
    example["circuito_recuperacao"]
)

full_circuit.measure_all()

compiled_circuit = transpile(full_circuit, simulator)

job = simulator.run(compiled_circuit, shots=1024)
counts = job.result().get_counts()

counts